# Tome 14, Chapter 5: Random multiplicative functions (Python companion)

Exercises first, solutions at the end. We generate Steinhaus and Rademacher random multiplicative functions up to $10^6$, check the exact fourth moment of Theorem 5.5, watch Wintner's theorem and the first moment drifting below the Gaussian value, test the conditional central limit theorem of Section 5.4, and compare the Steinhaus model with the Dirichlet polynomial $\sum n^{-it}$ at a random $t$. The computer science application: **generating a multiplicative function for all $n\le N$ at once**, by additive updates over prime powers, in $O(N\log\log N)$ operations and batched over samples. Requires `numpy`.

In [ ]:
import math, time, itertools
import numpy as np

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

N = 10**6
P = primes_upto(N)
powers = []                       # (prime power q, index of its prime)
for j, p in enumerate(P):
    q = int(p)
    while q <= N: powers.append((q, j)); q *= int(p)
squarefree = np.ones(N + 1, dtype=bool)
for p in P[P <= 1000]: squarefree[int(p)**2::int(p)**2] = False
rng = np.random.default_rng(14)

def steinhaus(M=N):
    # f(n) = exp(i theta(n)) with theta completely additive: add the angle of p at every multiple of p^k
    ang = rng.random(len(P)) * 2 * np.pi
    th = np.zeros(M + 1)
    for q, j in powers:
        if q > M: continue
        th[q::q] += ang[j]
    f = np.exp(1j * th); f[0] = 0
    return f

def rademacher(M=N):
    flip = rng.random(len(P)) < 0.5
    par = np.zeros(M + 1, dtype=np.int8)
    for j in np.nonzero(flip)[0]:
        p = int(P[j])
        if p > M: break
        par[p::p] ^= 1
    f = np.where(par == 1, -1.0, 1.0) * squarefree[:M + 1]; f[0] = 0
    return f

## Exercises

**P5.1 (★) Generating the models.** Time `steinhaus()` and `rademacher()` for $N=10^6$, check multiplicativity on random pairs, and estimate $\mathbb E|S(x)|^2/x$ from $100$ samples at $x=10^3,10^6$ (Proposition 5.2). Why is updating over prime powers cheaper than factoring every $n$?

**P5.2 (★) The fourth moment.** Check the formula $\mathbb E|S(x)|^4=\sum_{m\le x}c_m\lfloor x/m\rfloor^2$ against a brute-force count of $ab=cd$ for $x\le30$, then compute $\frac{1}{x^2}\mathbb E|S(x)|^4-\frac{12}{\pi^2}\log x$ for $x=10^2,\dots,10^7$. Does it converge?

**P5.3 (★★) Wintner's theorem.** For $20$ Rademacher samples, compute $\max_{x\le10^6}|S(x)|/\sqrt x$ and the last $x$ where $|S(x)|>\sqrt{x}\,\log\log x$. What does Theorem 5.4 say about these numbers, and what does it not say?

**P5.4 (★★) Better than square root.** With $300$ Steinhaus samples, estimate $\mathbb E|S(x)|/\sqrt x$ and $P(|S(x)|^2\ge\lfloor x\rfloor/2)$ for $x=10^2,10^4,10^6$ (Figure 5.3 and Exercise 5.2).

**P5.5 (★★) The conditional central limit theorem.** For $x=10^5$ and $500$ Steinhaus samples, compute $T(x)$ (the sum over $n$ with $P^+(n)>\sqrt x$), $V(x)$, and the moments $\mathbb E|W|^2,\mathbb E|W|^4$ of $W=T/\sqrt V$ (Gaussian values $1$ and $2$), and of $T/\sqrt{x\log2}$. Plot a histogram of $V(x)/x$.

**P5.6 (★★) The Steinhaus model is $n^{-it}$.** For $x=50$, compare $\frac1T\int_0^T|\sum_{n\le x}n^{-it}|^4\,dt$ (numerically, $T=10^5$) with $\mathbb E|S(x)|^4$ from P5.2. Explain the agreement with the independence of $\log p$ over $\mathbb Q$.

## Solutions

### P5.1

In [ ]:
t = time.time(); f = steinhaus(); t1 = time.time() - t
t = time.time(); g = rademacher(); t2 = time.time() - t
print(f"steinhaus {t1:.2f}s, rademacher {t2:.2f}s for N = {N}")
r = np.random.default_rng(0)
for _ in range(5):
    a, b = r.integers(1, 1000, 2)
    print(a, b, abs(f[a * b] - f[a] * f[b]) < 1e-9, g[a * b] == (g[a] * g[b] if math.gcd(int(a), int(b)) == 1 else g[a * b]))
xs = [10**3, 10**6]; acc = np.zeros(2)
for _ in range(100):
    S = np.cumsum(steinhaus()); acc += np.abs(S[xs])**2 / np.array(xs)
print("E|S(x)|^2/x estimates:", acc / 100)

The updates cost $\sum_{p^k\le N}N/p^k\approx N\log\log N$ additions, against about $N\sqrt N$ for trial division or $N\log N$ for factoring by a smallest-prime-factor table. The estimates of $\mathbb E|S(x)|^2/x=1$ are noisy: $|S(x)|^2/x$ has a heavy right tail (Theorem 5.5), so its sample mean converges slowly.

### P5.2

In [ ]:
def fourth_exact(x):
    phi = np.arange(x + 1, dtype=float)
    for p in P[P <= x]: phi[p::p] *= (1 - 1 / p)
    m = np.arange(1, x + 1); c = 2 * phi[1:]; c[0] = 1
    return float(np.sum(c * (x // m).astype(float) ** 2))
for x in [5, 10, 20, 30]:
    brute = sum(1 for a, b, c in itertools.product(range(1, x + 1), repeat=3) if (a * b) % c == 0 and a * b // c <= x)
    print(x, brute, fourth_exact(x))
P = primes_upto(10**7)
for k in range(2, 8):
    x = 10**k
    print(f"x=1e{k}:  E|S|^4/x^2 = {fourth_exact(x) / x**2:.4f},  minus (12/pi^2) log x = {fourth_exact(x) / x**2 - 12 / math.pi**2 * math.log(x):.4f}")
P = primes_upto(N)

The brute force enumerates $(a,b,c)$ and counts $d=ab/c$ when it is an integer $\le x$. The difference settles at a constant, about $-0.511$, as the $O(x^2)$ term of Theorem 5.5 allows: the formula is very accurate, while the sampled averages of Figure 5.2 are not.

### P5.3

In [ ]:
grid = np.arange(1, N + 1)
for k in range(20):
    S = np.cumsum(rademacher())[1:]
    ratio = np.abs(S) / np.sqrt(grid)
    big = np.nonzero(np.abs(S[15:]) > np.sqrt(grid[15:]) * np.log(np.log(grid[15:])))[0]
    last = grid[15:][big[-1]] if len(big) else None
    if k < 8: print(f"sample {k}: max |S|/sqrt x = {ratio.max():.2f} at x = {ratio.argmax() + 1},  last x with |S| > sqrt(x) loglog x: {last}")

The maxima are of order $1$ to $3$, attained at small or moderate $x$. Theorem 5.4 says that for each $\varepsilon$ the ratio $|S(x)|/x^{1/2+\varepsilon}$ is eventually below $1$; it says nothing about when. Harper's theorem says that $|S(x)|/\sqrt x$ is unbounded, but it grows like $(\log\log x)^{1/4}$, which no computer can see: the events $|S(x)|>\sqrt x\log\log x$ simply stop after small $x$ in most samples.

### P5.4

In [ ]:
xs = np.array([10**2, 10**4, 10**6]); m1 = np.zeros(3); pz = np.zeros(3); M = 300
for _ in range(M):
    S = np.cumsum(steinhaus())[xs]
    m1 += np.abs(S) / np.sqrt(xs); pz += (np.abs(S)**2 >= xs / 2)
print("E|S|/sqrt x:", m1 / M, "  Gaussian:", math.sqrt(math.pi) / 2)
print("P(|S|^2 >= x/2):", pz / M, "  Gaussian:", math.exp(-0.5))

The first moment drifts below $\frac{\sqrt\pi}2=0.886$, and the probability of a large value drifts below $e^{-1/2}=0.61$, slowly: Paley–Zygmund's $\frac c{\log x}$ is far from the truth. Both tend to $0$, but at the speed of a power of $\log\log x$.

### P5.5

In [ ]:
x = 10**5; r = int(math.isqrt(x))
big = P[(P > r) & (P <= x)]
Ws, Tn, Vs = [], [], []
for _ in range(500):
    f = steinhaus(x); S = np.cumsum(f)
    coef = S[x // big]                       # S(x/p) for p > sqrt(x)
    T = np.sum(f[big] * coef); V = np.sum(np.abs(coef)**2)
    Ws.append(T / math.sqrt(V)); Tn.append(T / math.sqrt(x * math.log(2))); Vs.append(V / x)
Ws, Tn, Vs = np.array(Ws), np.array(Tn), np.array(Vs)
print("W = T/sqrt(V):        E|W|^2 =", np.mean(np.abs(Ws)**2), " E|W|^4 =", np.mean(np.abs(Ws)**4))
print("T/sqrt(x log 2):      E|.|^2 =", np.mean(np.abs(Tn)**2), " E|.|^4 =", np.mean(np.abs(Tn)**4))
print("V/x: mean", Vs.mean(), " (log 2 =", math.log(2), ")  quantiles", np.quantile(Vs, [0.1, 0.5, 0.9]))

Normalised by its own random variance, $T$ has Gaussian moments $1$ and $2$ up to sampling error; normalised by the constant $\sqrt{x\log 2}$, its fourth moment is visibly larger than $2$. The variance $V(x)/x$ has mean $\log2$ but a wide, skewed distribution: its median is below its mean. This is the Gaussian mixture of Section 5.4. (With matplotlib, `plt.hist(Vs, 40)` shows it.)

### P5.6

In [ ]:
x = 50; T = 10**5
t = np.linspace(0, T, 2_000_001)
logs = np.log(np.arange(1, x + 1))
D = np.zeros_like(t, dtype=complex)
for L in logs: D += np.exp(-1j * t * L)
print("time average of |sum n^{-it}|^4:", np.mean(np.abs(D)**4), "  E|S(50)|^4 =", fourth_exact(50))

The two numbers agree to four significant digits. Expanding $|\sum n^{-it}|^4$ gives $\sum_{a,b,c,d}(cd/ab)^{it}$, whose time average tends to the number of solutions of $ab=cd$, since $\frac1T\int_0^Te^{it\log r}dt\to0$ for every rational $r\ne1$. This is the same count as for the Steinhaus model: $(p^{-it})_p$ behaves like independent uniform points on the circle because the numbers $\log p$ are linearly independent over $\mathbb Q$ (unique factorisation). Chapter 6 makes this precise (Kronecker–Weyl).